# Food in printed Dutch texts, 1550–1920
### A hands-on notebook: the words that keep company with food

In this notebook you explore **which words appear around food words** in printed Dutch texts (cookbooks, herbals, household books, novels, songs, travel accounts), and whether that company **differs between kinds of text and changes over time**.

**You do not need to know how to program.** Every step is a text cell (like this one) followed by a code cell. Your job is to:

1. **Read** the explanation.
2. **Run** the code cell below it (click it, then **Shift + Enter**, or the ▶ button).
3. **Change** the values in the cells marked ✏️ (only what is between the quotes) and run them again.

The notebook has three parts, which you can run separately after Step 1:

- **Part A · The company food words keep**: collocations, over time and between kinds of use (Steps 1–8)
- **Part B · Recipe legacy**: passages reused between works (Steps B1–B2)
- **Part D · A long view: Indies names, 1870–1990** (cookbooks and newspaper recipes)
- **Part C · Caveats and playground**

✏️ = change something here · 💡 = how to read what you see · ⚠️ = be careful before drawing conclusions

## The big idea in two minutes

**"You shall know a word by the company it keeps."** (J. R. Firth, 1957) Words used in similar contexts tend to have similar meanings. If *suiker* keeps appearing near *koken*, *pond* and *boter* in one kind of book, and near *genezen*, *drinken* and *maag* in another, that tells you something about what sugar *was* in those texts.

We measure that company by **counting**: for every food mention we look at the five words to its left and right (its **collocates**) and ask which of them occur near it more often than you would expect.

**Where the data comes from.** A research pipeline searched printed Dutch texts (mostly DBNL) for food words, cut a window of about 180 characters on either side, and had a language model label each mention with what is happening with the food:

| label (`frame`) | meaning |
|---|---|
| COOKING_CREATION | food is prepared, cooked, baked |
| INGESTION | food is eaten or drunk |
| CURE | food used as medicine |
| PRESERVING | food is kept, pickled, candied, salted |
| NONE / dropped | a food word, but no food event (or not about food at all) |

💡 **Why counting, not word embeddings?** Embeddings (word vectors) need many millions of words *per period*. Our harvest has about 2.5 million words *in total*, unevenly spread over four centuries. Counting collocates works at this scale; embeddings would mostly measure noise.

---
# Part A · The company food words keep

Steps 1 to 8: load the data, look at the material, read concordances, count collocates, follow them over time and compare kinds of use. **Step 1 is needed for Parts B and C too.**

---
## Step 1 · Load the tools and the data

We use **pandas** (tables) and **matplotlib** (plots). Both come with most Python installations, including Google Colab. The data files (`mentions.parquet`, `works.csv`) must be in the same folder as this notebook (✏️ change `DATA_DIR` if they are elsewhere).

In [ ]:
import math, re
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.dpi"] = 100

# ✏️ Folder with mentions.parquet and works.csv
DATA_DIR = Path(".")

df = pd.read_parquet(DATA_DIR / "mentions.parquet")
works = pd.read_csv(DATA_DIR / "works.csv")
print(f"✅ Loaded {len(df):,} food mentions from {len(works):,} works.")

---
## Step 2 · What is in the harvest?

Before looking at any word, look at the **material**: which kinds of text, and how much per period. This matters more here than in most corpora.

The first table shows, per kind of text (`genre`), how many mentions there are and how they were labelled. The plot shows how many **works** we have per 25 years.

In [ ]:
overview = pd.crosstab(df["genre"], df["status"], margins=True, margins_name="total")
display(overview)

dated = works.dropna(subset=["year"])
ax = (dated.assign(period=(dated["year"] // 25 * 25).astype(int))
           .groupby(["period", "genre"]).size().unstack(fill_value=0)
           .plot(kind="bar", stacked=True, figsize=(12, 4), width=0.85))
ax.set_title("Works in the harvest per 25 years (edition year)")
ax.set_xlabel(""); ax.set_ylabel("works")
plt.tight_layout(); plt.show()

💡 **What you see:** very few works before 1650, a lot after 1750. Genre `UNKNOWN` is mostly literature, children's books and songs (the catalogue could not tell).

⚠️ **This is a sample, very unevenly spread.** In thin periods a single book decides the picture. One household dictionary (Chomel, *Huishoudelyk woordboek*, 1743) alone has thousands of mentions. Absence of a word in a period is not evidence that the word was not used: often we simply have no material.

---
## Step 3 · Is my food word in the data?

Spelling varied enormously (*suiker, suycker, suijcker, zuiker*). The harvest pools spelling variants under one **normalised term** (`pref_label`). The table below shows, for the terms you type, how many mentions and how many different works there are per 50 years.

### ✏️ Try it
Type normalised terms between the quotes, separated by commas. Not sure of the term? The second part of the cell lists the most frequent ones.

In [ ]:
# ✏️ Which food terms?
MY_TERMS = "suiker, thee, koffie, peper, brood"

PERIOD = 50   # ✏️ length of a period in years

def terms(text):
    return [t.strip().lower() for t in text.split(",") if t.strip()]

def coverage(term_list, period=PERIOD):
    sub = df[df["pref_label"].isin(term_list) & df["year"].notna()].copy()
    sub["period"] = (sub["year"] // period * period).astype(int)
    m = sub.pivot_table(index="pref_label", columns="period", values="mention_id", aggfunc="size", fill_value=0)
    w = sub.pivot_table(index="pref_label", columns="period", values="work_id", aggfunc="nunique", fill_value=0)
    return m.astype(str) + " (" + w.astype(str) + ")"

display(Markdown("**Mentions (works) per period**"))
display(coverage(terms(MY_TERMS)))
display(Markdown("**The 40 most frequent normalised terms**"))
print(", ".join(df["pref_label"].value_counts().head(40).index))

💡 **Reading tip:** "120 (3)" means 120 mentions, but from only 3 works. That is three books talking, not a period. Prefer cells with many works.

---
## Step 4 · Read the mentions: a concordance

Statistics are only as good as your reading of the texts behind them. A **concordance** (KWIC, *keyword in context*) shows every mention on one line with the food word in the middle.

### ✏️ Try it
Choose a term, optionally a frame (or `""` for all), a period, and how many lines.

In [ ]:
# ✏️ What to show
TERM = "suiker"
FRAME = "CURE"          # COOKING_CREATION, INGESTION, CURE, PRESERVING, NONE, or "" for all
FROM_YEAR, TO_YEAR = 1600, 1800
LINES = 15

def concordance(term, frame="", start=1500, end=1950, n=15, width=60, seed=0):
    sub = df[(df["pref_label"] == term) & df["year"].between(start, end)]
    if frame:
        sub = sub[sub["frame"] == frame]
    if sub.empty:
        print("No mentions for this choice."); return
    sub = sub.sample(min(n, len(sub)), random_state=seed).sort_values("year")
    for r in sub.itertuples():
        print(f"{int(r.year):>4}  {r.left[-width:]:>{width}} [{r.keyword}] {r.right[:width]}")
    print(f"\n({len(sub)} shown; titles: {', '.join(sorted(set(sub['title'].str[:30])))[:300]})")

concordance(TERM, FRAME, FROM_YEAR, TO_YEAR, LINES)

---
## Step 5 · The company a word keeps: collocates

Now we count. For a set of mentions (say: *suiker* in CURE contexts) we collect all words within five positions to the left and right, and score each word with **log-Dice**:

> log-Dice = 14 + log₂( 2 × (times the word occurs near our mentions) ÷ (number of our mentions + times the word occurs anywhere in the harvest) )

- The maximum is **14**; higher = a more exclusive companion.
- It does not reward words just for being frequent everywhere, and it is stable across subsets of different size.

Three safeguards are built in:
- **Spelling pooling** (`norm`): *suycker* and *suiker* count as one word.
- A small list of **function words** (*de, het, van, ende*…) is ignored, otherwise they top every list.
- **At most 50 mentions per work** (`PER_WORK_CAP`), so one big book cannot dominate the result.

In [ ]:
TOKEN = re.compile(r"[^\W\d_]+", re.UNICODE)
SPELL = [("uij", "ui"), ("uy", "ui"), ("ij", "y"), ("ck", "k"), ("ae", "a"), ("gh", "g"), ("ph", "f")]
STOP = frozenset("""de het een en van in op te met die dat is aan als of voor door men zyn syn zijn
hy hij sy zij ze ik gy gij wy wij u uw hun haar zich niet maar ook so zo dan want om tot by bij na
uyt uit over onder al alle dit dese deze den der des eens wel noch nog seer zeer veel wat wie waer
waar daer daar dar hier was waren wert wordt worden werd heeft hebben had hadden can kan sal zal souden
zouden moet mogen mag eenen eene ende oft ofte hem hen haer ons onse wort ofte nbsp""".split())
PER_WORK_CAP = 50   # ✏️ at most this many mentions per work

def norm(token):
    t = token.lower()
    for a, b in SPELL:
        t = t.replace(a, b)
    return t

def window(row, k=5):
    left = [norm(t) for t in TOKEN.findall(row.left)][-k:]
    right = [norm(t) for t in TOKEN.findall(row.right)][:k]
    return [t for t in left + right if t not in STOP and len(t) > 2]

def cap(data, per_work=PER_WORK_CAP, seed=0):
    return pd.concat([g.sample(min(len(g), per_work), random_state=seed) for _, g in data.groupby("work_id")]) \
        if len(data) else data

REFERENCE = Counter(t for w in df.apply(window, axis=1) for t in w)

def collocates(subset, top=20, min_freq=4):
    subset = cap(subset)
    co = Counter(t for w in subset.apply(window, axis=1) for t in w)
    rows = [{"collocate": t, "times near": f, "times in harvest": REFERENCE[t],
             "log-Dice": round(14 + math.log2(2 * f / (len(subset) + REFERENCE[t])), 2)}
            for t, f in co.items() if f >= min_freq]
    if not rows:
        return pd.DataFrame(columns=["collocate", "times near", "times in harvest", "log-Dice"])
    return pd.DataFrame(rows).sort_values("log-Dice", ascending=False).head(top).reset_index(drop=True)

print("✅ Collocation tools ready.")

### ✏️ Try it
Choose a term and a frame (or `""` for all frames).

In [ ]:
# ✏️
TERM = "suiker"
FRAME = "CURE"

sel = df[(df["pref_label"] == TERM) & ((df["frame"] == FRAME) if FRAME else True)]
display(Markdown(f"**Collocates of _{TERM}_ ({FRAME or 'all frames'}), {len(sel):,} mentions**"))
display(collocates(sel))

💡 **What to look for:** do the companions form a theme (cooking utensils, quantities, illnesses, drinks)? Try the same term with another frame and compare.

⚠️ Below about 30 mentions the list is shaky: a few sentences decide it.

---
## Step 6 · Did the company change over time?

Now we put the collocate lists of several periods **side by side**. Old companions drop out, new ones come in. Words that occur in **every** period shown are printed **bold blue**: the stable core.

### ✏️ Try it
Choose a term, the periods (start years), and the period length.

In [ ]:
# ✏️
TERM = "thee"
PERIODS = [1650, 1700, 1750, 1800, 1850]
PERIOD_LENGTH = 50
TOP = 12

def collocates_over_time(term, periods, length=50, top=12):
    cols, sizes = {}, {}
    for p in periods:
        sub = df[(df["pref_label"] == term) & df["year"].between(p, p + length - 1)]
        sizes[p] = (len(sub), sub["work_id"].nunique())
        words = list(collocates(sub, top=top)["collocate"]) if len(sub) else []
        cols[f"{p}-{p + length - 1}"] = words + [""] * (top - len(words))
    table = pd.DataFrame(cols, index=range(1, top + 1))
    return table, sizes

def highlight_shared(table):
    sets = [set(v for v in table[c] if v) for c in table.columns if any(table[c])]
    shared = set.intersection(*sets) if sets else set()
    return table.style.map(lambda v: "font-weight: bold; color: #1f77b4" if v in shared else "")

table, sizes = collocates_over_time(TERM, PERIODS, PERIOD_LENGTH, TOP)
display(Markdown("Mentions (works) per period: " +
                 ", ".join(f"{p}: {n} ({w})" for p, (n, w) in sizes.items())))
display(highlight_shared(table))

💡 **Reading tip:** a period with a very different list may just rest on a few works. Check the "(works)" numbers above the table first. Change in the *kind of books* we have (more novels after 1800, more medical books around 1700) changes the company too; that is not the same as a change in how people used the food.

---
## Step 7 · Compare two kinds of use

What distinguishes sugar-as-medicine from sugar-in-the-kitchen? We compute collocates for two groups and show the words that score much higher in one than in the other.

### ✏️ Try it
Define two groups by frame (and optionally genre). Leave a term empty (`""`) to compare all food words.

In [ ]:
# ✏️ Group A and group B
TERM = "suiker"
A = dict(frame="CURE", genre="")
B = dict(frame="COOKING_CREATION", genre="")

def group(term, frame, genre):
    sub = df
    if term:  sub = sub[sub["pref_label"] == term]
    if frame: sub = sub[sub["frame"] == frame]
    if genre: sub = sub[sub["genre"] == genre]
    return sub

ga, gb = group(TERM, **A), group(TERM, **B)
ca = collocates(ga, top=300, min_freq=3).set_index("collocate")["log-Dice"]
cb = collocates(gb, top=300, min_freq=3).set_index("collocate")["log-Dice"]
both = pd.concat({"A": ca, "B": cb}, axis=1).fillna(0)
both["A minus B"] = both["A"] - both["B"]
name_a, name_b = A["frame"] or "all", B["frame"] or "all"
display(Markdown(f"**More typical of A ({name_a}, {len(ga)} mentions)**"))
display(both.sort_values("A minus B", ascending=False).head(12))
display(Markdown(f"**More typical of B ({name_b}, {len(gb)} mentions)**"))
display(both.sort_values("A minus B").head(12))

⚠️ **The selection shows through.** Many mentions were selected *because* a food word stood near a food verb (*koken, eten, drinken*), and the labels follow those verbs. So verbs will separate the groups almost by design. Look past them, at nouns and adjectives (illnesses, utensils, quantities, places, people).

---
## Step 8 · A baseline: how much do lists differ anyway?

To judge whether a difference is interesting, compare with a word you expect to be **stable**, such as *brood* (bread) or *water*. If its company changes just as much over time, the change you found for your word may be mostly noise or changing material.

In [ ]:
# ✏️ Baseline term (run with the same PERIODS as Step 6)
BASELINE = "brood"

table, sizes = collocates_over_time(BASELINE, PERIODS, PERIOD_LENGTH, TOP)
display(Markdown("Mentions (works) per period: " +
                 ", ".join(f"{p}: {n} ({w})" for p, (n, w) in sizes.items())))
display(highlight_shared(table))

---
# Part B · Recipe legacy

Recipes were copied, adapted and reprinted, and so were remedies. Part B looks for that legacy directly: **the same passage turning up in different works**. It needs only Step 1 (and the spelling helper of Step 5, which it defines again below if you skip Part A).

## Step B1 · Find reused passages

Recipes were copied, adapted and reprinted. When two works contain the **same passage**, almost word for word, we can see that legacy. Here we compare the contexts of recipe and medical works: two contexts from *different* works that share many runs of five words (spelling pooled) are probably the same text.

✏️ `MIN_OVERLAP` is the share of five-word runs two contexts must share (0.3 = 30%).

In [ ]:
# Defined in Step 5; repeated here so Part B runs on its own
TOKEN = re.compile(r"[^\W\d_]+", re.UNICODE)
SPELL = [("uij", "ui"), ("uy", "ui"), ("ij", "y"), ("ck", "k"), ("ae", "a"), ("gh", "g"), ("ph", "f")]
def norm(token):
    t = token.lower()
    for a, b in SPELL:
        t = t.replace(a, b)
    return t

# ✏️
GENRES = ["RECIPE_PRACTICE", "MEDICAL"]
MIN_OVERLAP = 0.3

def shingles(text, n=5):
    w = [norm(t) for t in TOKEN.findall(text)]
    return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}

pool = df[df["genre"].isin(GENRES)].drop_duplicates(["work_id", "context"]).reset_index(drop=True)
sh = [shingles(c) for c in pool["context"]]
index = defaultdict(set)
for i, s in enumerate(sh):
    for g in s:
        index[g].add(i)
shared = Counter()
for g, ids in index.items():
    if 1 < len(ids) < 30:
        ids = sorted(ids)
        for a in ids:
            for b in ids:
                if a < b and pool.at[a, "work_id"] != pool.at[b, "work_id"]:
                    shared[(a, b)] += 1
pairs = [(a, b, n / max(1, min(len(sh[a]), len(sh[b])))) for (a, b), n in shared.items()]
pairs = [p for p in pairs if p[2] >= MIN_OVERLAP]

def label(i):
    r = pool.loc[i]
    return f"{r.title[:45]} ({'' if pd.isna(r.year) else int(r.year)})"

links = Counter(tuple(sorted((label(a), label(b)))) for a, b, _ in pairs)
display(Markdown(f"**{len(pairs)} reused passages between different works.** Most connected pairs:"))
display(pd.DataFrame([{"work 1": k[0], "work 2": k[1], "shared passages": v}
                      for k, v in links.most_common(12)]))
a, b, s = max(pairs, key=lambda p: p[2])
display(Markdown(f"**Example** (overlap {s:.0%}):\n\n> {label(a)}: …{pool.at[a, 'context'][:220]}…\n\n"
                 f"> {label(b)}: …{pool.at[b, 'context'][:220]}…"))

## Step B2 · Reading the result

💡 **What you see:** mostly **editions of the same work** (the Amsterdam and Leeuwarden pharmacopoeias reprinted passages almost unchanged across decades).

⚠️ **Why so few links between different titles?** The harvest holds *windows* around food words, sampled per work, not whole books. Reuse only shows where both works were sampled at the same spot. Real lineage questions (Chomel's *Huishoudelyk woordboek* and its *Vervolg*, the *Hollandsche* and *Geldersche keuken-meid*) need the full texts: a separate study.

---
# Part D · A long view: Indies names, 1870–1990

Many ingredients have two names: a Dutch one (*koriander*, *Spaansche peper*, *kokosmelk*) and a Malay one used in the Indies (*ketoembar*, *lombok*, *santen*). **Which name a text uses tells you whose kitchen it speaks from**, and when the Malay name reaches Dutch kitchens.

Part D uses one small table, `names_over_time.csv`, built from two sources:

| source | years | what was counted |
|---|---|---|
| **cookbooks** (full text) | 1790–1940 | mentions of each name per book; includes three editions of the *Oost-Indisch kookboek* (Samarang 1870, 1893, 1913), the *Indisch kookboek* (Nijmegen 1872) and Dutch cookbooks |
| **newspaper recipes** (van Erp, Wevers & Huurdeman 2018, via the *Dutch-historical-recipe-trends* repository) | 1940s–1990s | recipes whose ingredient list mentions each name, per decade |

💡 We do not compare *how often* the ingredient occurs (a cookbook and a newspaper recipe are different kinds of text), but **the share of the Malay name**: Malay ÷ (Dutch + Malay). That ratio says which name writers chose, whatever the genre.

It needs only Step 1's tools (pandas, matplotlib); run it on its own if you like.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path(".")   # ✏️ folder with names_over_time.csv
names = pd.read_csv(DATA_DIR / "names_over_time.csv")
names["mentions"] = names["dutch"] + names["malay"]
names["malay_share"] = names["malay"] / names["mentions"]
INDIES = ("Oost-Indisch", "Indisch kookboek", "Visser")
names["kind"] = names.apply(lambda r: "newspaper recipes" if r.source != "cookbook"
                            else ("Indies cookbook" if r.unit.startswith(INDIES) or "Indisch" in r.unit
                                  else "Dutch cookbook"), axis=1)
print("Concepts:", ", ".join(names["concept"].unique()))

## Step D1 · The table per concept

✏️ Choose a concept. Each row is one book (with its year) or one decade of newspaper recipes. `mentions` is how often either name occurs; rows with very few mentions say little.

In [ ]:
# ✏️
CONCEPT = "coriander"

view = names[names["concept"] == CONCEPT][["kind", "unit", "year", "dutch", "malay", "mentions", "malay_share"]]
view[view["mentions"] > 0].sort_values("year").round(2)

## Step D2 · The share of the Malay name over time

One small plot per concept. **Dots** are books (filled = Indies cookbook, open = Dutch cookbook), the **line** is the newspaper recipes per decade. Points resting on fewer than `MIN_MENTIONS` mentions are left out.

✏️ Change the concepts or `MIN_MENTIONS`.

In [ ]:
# ✏️
CONCEPTS = ["coriander", "chili", "coconut milk", "onion/shallot"]
MIN_MENTIONS = 5

fig, axes = plt.subplots(1, len(CONCEPTS), figsize=(4 * len(CONCEPTS), 3.4), sharey=True)
for ax, concept in zip(axes, CONCEPTS):
    d = names[(names["concept"] == concept) & (names["mentions"] >= MIN_MENTIONS)]
    ind, dut, news = (d[d["kind"] == k] for k in ("Indies cookbook", "Dutch cookbook", "newspaper recipes"))
    ax.scatter(ind["year"], ind["malay_share"], color="#0072B2", label="Indies cookbook", zorder=3)
    ax.scatter(dut["year"], dut["malay_share"], facecolors="none", edgecolors="#D55E00", label="Dutch cookbook", zorder=3)
    ax.plot(news["year"] + 5, news["malay_share"], color="#555555", marker="s", ms=4, label="newspaper recipes")
    ax.set_title(concept); ax.set_ylim(-0.05, 1.05); ax.set_xlim(1860, 2000); ax.grid(alpha=0.3)
axes[0].set_ylabel("share of the Malay name")
axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
plt.tight_layout(); plt.show()

💡 **What to look for:**

- In the **Indies cookbooks** (Samarang) the Malay name dominates for coriander, chili and coconut milk, and it is stable over three editions (1870, 1893, 1913): a convention, not a drift.
- The **Indies cookbook printed in Nijmegen (1872)** writes *Spaansche peper* where Samarang writes *lombok*: a book for readers in the Netherlands translates the market word.
- Around **1938–1940** *ketoembar* and *santen* appear in Dutch cookbooks.
- In the **newspaper recipes after the war** the Dutch names are back (*koriander*, *kokosmelk*); *ketoembar* appears mainly as a gloss, e.g. "gemalen koriander ketoembar". (*Ketjap* is not in the table: soy sauce simply *was* ketjap, there is no Dutch rival name to compare with.)

⚠️ **Be careful:**

- **Few books before 1940**, and the cookbook years are edition years.
- **The newspaper data are extracted ingredient lists,** not full recipes: a name that the extraction tool split off or dropped is not counted, and a gloss counts as a mention of both names.
- **Pattern matching:** words were matched on their beginning; homonyms are possible (*lombok* the island, *laos* the country), though rare in recipes.

## Step D3 · Try other concepts

The table also has `bean sprouts`, `ginger`, `cloves`, `cinnamon`, `coconut`. The classic trade spices (ginger, cloves, cinnamon) stay Dutch everywhere, even in the Indies cookbooks: compare them with coriander. Why might that be?

✏️ Put them in `CONCEPTS` in Step D2 and run it again.

---
# Part C · Caveats and playground

## Step C1 · How much can we trust this?

1. **A sample, unevenly spread** (Step 2). Thin periods are a few books; one big book can dominate. Use the per-work cap and look at the number of works.
2. **The selection is not neutral.** Mentions were mostly picked near food verbs; compare groups *within* the harvest, not with Dutch in general.
3. **Windows overlap.** Two mentions close together share text, so a companion can be counted twice.
4. **Spelling pooling is crude.** `norm()` merges common variants but not all; check the concordance.
5. **Labels are model output.** Frames come from a language model: a strong hint, not ground truth. Genre is an inference per work (UNKNOWN is mostly literature).
6. **Dates are edition years:** when a text was printed, not when it was compiled or the practice began.

## Step C2 · Playground

Change the values and run the cell again. You cannot break anything; if something goes wrong, undo your change (Ctrl/Cmd + Z).

The playground uses the tools of Part A (Steps 4 and 5): run those first.

In [ ]:
# ✏️ Change these and re-run
my_term = "koffie"
my_frame = ""                 # "" = all frames
my_from, my_to = 1700, 1900

display(Markdown(f"**Concordance: _{my_term}_**"))
concordance(my_term, my_frame, my_from, my_to, 10)
sel = df[(df["pref_label"] == my_term) & df["year"].between(my_from, my_to)]
if my_frame:
    sel = sel[sel["frame"] == my_frame]
display(Markdown(f"**Collocates ({len(sel)} mentions)**"))
display(collocates(sel, top=15))

💡 **Before you present a finding,** be ready to say: (1) what the companions show, (2) for which kind of text and period, (3) how many *works* it rests on, and (4) what could make it unreliable (selection, sample, spelling, labels).